In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

root = '/home/amenacer/Stage/base_de_donnees/rats'
essais = [
    '2-essaie-data11/img-segmented',
    '3-essaie-data12/img-segmented',
    '4-essaie-data13/img-segmented'
]
rats = [f'rat{i}.nii.gz' for i in range(1, 30)]

def load_mask(path):
    return nib.load(path).get_fdata().astype(int)

def dice(mask1, mask2):
    intersection = np.sum((mask1 > 0) & (mask2 > 0))
    s = np.sum(mask1 > 0) + np.sum(mask2 > 0)
    return 2 * intersection / (s + 1e-8)

pdf = PdfPages('comparaison_segmentations_2DplusT.pdf')
summary = []

for rat in rats:
    masks = []
    missing = False
    for essai in essais:
        file_path = os.path.join(root, essai, rat)
        if not os.path.exists(file_path):
            print(f"File missing: {file_path}")
            missing = True
            break
        masks.append(load_mask(file_path))
    if missing:
        continue

    if masks[0].ndim != 3:
        print(f"{rat} n'est pas 2D+T, saute")
        continue

    nb_frames = masks[0].shape[2]
    for t in range(nb_frames):
        fig, axes = plt.subplots(1, 7, figsize=(30, 5))
        # Affichage des masques bruts pour contrôle
        axes[0].imshow(masks[0][:,:,t], cmap='gray')
        axes[0].set_title(f'data11\nFrame {t}')
        axes[0].axis('off')
        axes[1].imshow(masks[1][:,:,t], cmap='gray')
        axes[1].set_title(f'data12\nFrame {t}')
        axes[1].axis('off')
        axes[2].imshow(masks[2][:,:,t], cmap='gray')
        axes[2].set_title(f'data13\nFrame {t}')
        axes[2].axis('off')
        # Différences entre les trois paires
        diff_11_12 = masks[0][:,:,t] != masks[1][:,:,t]
        diff_11_13 = masks[0][:,:,t] != masks[2][:,:,t]
        diff_12_13 = masks[1][:,:,t] != masks[2][:,:,t]
        axes[3].imshow(diff_11_12, cmap='hot')
        axes[3].set_title('11 vs 12')
        axes[3].axis('off')
        axes[4].imshow(diff_11_13, cmap='hot')
        axes[4].set_title('11 vs 13')
        axes[4].axis('off')
        axes[5].imshow(diff_12_13, cmap='hot')
        axes[5].set_title('12 vs 13')
        axes[5].axis('off')

        # Dice scores
        dice_11_12 = dice(masks[0][:,:,t], masks[1][:,:,t])
        dice_11_13 = dice(masks[0][:,:,t], masks[2][:,:,t])
        dice_12_13 = dice(masks[1][:,:,t], masks[2][:,:,t])
        # Résumé pour tableau
        summary.append([rat, t, dice_11_12, dice_11_13, dice_12_13])

        # Affichage Dice sur la figure
        axes[6].axis('off')
        axes[6].text(0, 0.7, f"Dice 11-12: {dice_11_12:.3f}\nDice 11-13: {dice_11_13:.3f}\nDice 12-13: {dice_12_13:.3f}", fontsize=18)

        pdf.savefig(fig)
        plt.close(fig)

if len(summary) > 0:
    df = pd.DataFrame(summary, columns=['rat', 'frame', 'Dice_11-12', 'Dice_11-13', 'Dice_12-13'])
    fig, ax = plt.subplots(figsize=(16,12))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center')
    pdf.savefig(fig)
else:
    print("Aucun résultat à afficher dans le tableau final.")

pdf.close()
print("PDF sauvegardé : comparaison_segmentations_2DplusT.pdf")
